# 08 — Explainable AI (SHAP)

SHAP TreeExplainer untuk RF/XGBoost classifier.
Output: top-N feature contribution per prediksi.
Ditampilkan di UI sebagai bar chart "kenapa wallet ini diklasifikasi X".

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import shap
shap.initjs()

In [ ]:
# Train a model
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=500, n_features=15, n_informative=10, random_state=42)
feat_names = ['tx_count', 'active_days', 'native_in', 'native_out', 'gas_spent',
              'unique_cp', 'dex_ratio', 'failed_ratio', 'self_ratio',
              'night_ratio', 'weekend_ratio', 'stablecoin_ratio',
              'avg_tx_value', 'std_tx_value', 'max_tx_value']
X_df = pd.DataFrame(X, columns=feat_names)
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42).fit(X_train, y_train)
print(f'Accuracy: {model.score(X_test, y_test):.3f}')

In [ ]:
# SHAP TreeExplainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
print(f'SHAP values shape: {np.array(shap_values).shape}')  # (n_classes, n_samples, n_features)

In [ ]:
# Summary plot (class 0)
shap.summary_plot(shap_values[0], X_test, feature_names=feat_names, show=False)
plt.title('SHAP Summary — Class 0'); plt.show()

In [ ]:
# Waterfall plot — explain single prediction
sample_idx = 0
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[0][sample_idx],
        base_values=explainer.expected_value[0],
        data=X_test.iloc[sample_idx].values,
        feature_names=feat_names,
    ),
    show=False,
)
plt.title(f'Waterfall — Sample {sample_idx}, Class 0'); plt.tight_layout(); plt.show()

In [ ]:
# Feature importance bar chart
shap.plots.bar(shap.Explanation(
    values=shap_values[0], base_values=explainer.expected_value[0],
    data=X_test.values, feature_names=feat_names,
), show=False)
plt.title('Mean |SHAP| per Feature'); plt.show()

## Integrasi dengan API

Endpoint `/explain` memanggil SHAP explainer, return top-N features:

```json
{
  "predicted_class": "dex_trader",
  "top_features": [
    {"feature": "dex_swap_ratio", "contribution": 0.32, "direction": "positive"},
    {"feature": "unique_tokens", "contribution": 0.18, "direction": "positive"}
  ]
}
```